# Tool calling basics

The simplest possible example of "agentic tool calling": the model can't do math reliably on its own for arbitrary numbers, so we give it one tool, `add(a, b)`, and let it decide when to call it.

**The loop, at a high level:**
1. We send a question + a description of the available tool(s) (the *tool schema*).
2. Instead of a plain text answer, the model can respond with a `tool_use` block: "call this tool, with these arguments."
3. We (the code, not the model) actually run the real function.
4. We send the result back to the model as a `tool_result`.
5. The model reads the result and gives a final text answer.

The model never executes anything itself — it only ever *asks* for a tool to be run. All execution happens in our own Python code. This is what makes tool calling safe-by-construction: the model can request `add(2, 2)` but it can't request `os.system("rm -rf /")` unless we've defined and exposed that as a tool ourselves.

We're writing this as a **manual loop** (not using the SDK's higher-level "tool runner" helper) specifically so every step of the mechanism is visible.

In [1]:
import os
import json

import anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env into the environment

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically
MODEL = "claude-opus-5"

## Step 1: define the tool

A tool has two halves that have to stay in sync:

- **The real Python function** — what actually runs when the tool is called.
- **The tool schema** — a JSON description of the tool's name, purpose, and expected inputs, sent to the model. This is the *only* thing the model ever sees; it never sees your Python source. It's reading `description` and `input_schema` and deciding, from the wording alone, whether and how to call it.

In [2]:
def add(a, b):
    return a + b


tools = [
    {
        "name": "add",
        "description": "Add two numbers together and return the sum.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"},
            },
            "required": ["a", "b"],
        },
    }
]

## Step 2: send the first request

We ask a question that needs the tool, and pass `tools=tools` so the model knows `add` is available. Note we're not calling `add()` ourselves here — we're just asking the model a question.

### `client.messages.create(...)`

This is the actual API call — it sends one request to Claude and gets one response back. The API is stateless, so every call is self-contained; nothing is remembered between calls except what you explicitly pass in `messages`.

Arguments used in this notebook:
- **`model`** — which model to call (e.g. `"claude-opus-5"`).
- **`max_tokens`** — the max number of tokens the model is allowed to generate in its reply. Not a target length, just a hard ceiling — if the model is mid-thought when it hits this, the response gets cut off (`stop_reason == "max_tokens"`).
- **`tools`** — the toolbox list (schemas for every tool the model may call this turn). Must be passed on every call, not just the first, since the model doesn't remember it either.
- **`messages`** — the full conversation history so far, as a list of `{"role": ..., "content": ...}` turns. Since the API is stateless, this is how the model "remembers" anything — including its own prior tool call.

It returns a `Message` object — the two fields we care about here are `.content` (list of content blocks: text, `tool_use`, etc.) and `.stop_reason` (why the model stopped: `"end_turn"` = done, `"tool_use"` = it wants a tool run, `"max_tokens"` = got cut off).

In [3]:
user_question = "What is 482193 + 917364?"

messages = [{"role": "user", "content": user_question}]

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("stop_reason:", response.stop_reason)
print(response.content)

stop_reason: tool_use
[TextBlock(citations=None, text="I'll add those numbers for you.", type='text'), ToolUseBlock(id='toolu_01AxDZM9DGY4ZH2C3Ust2d6C', caller=DirectCaller(type='direct'), input={'a': 482193, 'b': 917364}, name='add', type='tool_use', toolset_name=None)]


## Step 3: look at what came back

If `stop_reason` is `"tool_use"`, the model has decided to call a tool — it hasn't given a final answer yet, it's requesting a tool run first (it will give the final answer next, after seeing the result). `response.content` is a list of content blocks; one of them has `type == "tool_use"`, with:
- `.name` — which tool it wants (`"add"`)
- `.input` — the arguments it decided on, as a dict (e.g. `{"a": 482193, "b": 917364}`)
- `.id` — an ID we must echo back so the model knows which call this result belongs to (matters once there's more than one tool call in flight)

In [4]:
tool_use_block = next(b for b in response.content if b.type == "tool_use")

print("tool name:", tool_use_block.name)
print("tool input:", tool_use_block.input)
print("tool_use id:", tool_use_block.id)

tool name: add
tool input: {'a': 482193, 'b': 917364}
tool_use id: toolu_01AxDZM9DGY4ZH2C3Ust2d6C


## Step 4: actually run the tool

This is the one step that is 100% our own code — the model has no way to run this itself. We look at `tool_use_block.name` to decide which real function to call, then call it with the arguments the model provided.

In [5]:
if tool_use_block.name == "add":
    result = add(**tool_use_block.input)
else:
    raise ValueError(f"Unknown tool: {tool_use_block.name}")

print("real result:", result)

real result: 1399557


## Step 5: send the result back

Two things get appended to `messages`:
1. The assistant's own turn (`response.content`, including the `tool_use` block) — the API is stateless, so the model only knows what it "said" if we hand its own prior turn back to it.
2. A `user`-role message containing a `tool_result` block, with `tool_use_id` matching the original call and `content` set to the real result.

Then we call `messages.create` again with this updated history.

In [6]:
messages.append({"role": "assistant", "content": response.content})
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": str(result),
            }
        ],
    }
)

final_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("stop_reason:", final_response.stop_reason)
final_text = next(b.text for b in final_response.content if b.type == "text")
print("final answer:", final_text)

stop_reason: end_turn
final answer: 482193 + 917364 = **1,399,557**


## Recap

The whole mechanism, end to end: schema in → `tool_use` block out → we execute → `tool_result` back in → final text out. Everything else in "agentic tool calling" (multiple tools, multiple rounds, parallel tool calls, a database instead of `add`) is this same loop repeated, just with more tools and a `while` loop around steps 2–5 instead of doing it once by hand.

**Next step:** wrap steps 2–5 in a `while response.stop_reason == "tool_use":` loop so it keeps going until the model is done — then swap `add` for a real database query tool.